In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import json
import h5py
import os
import cv2
from tqdm import tqdm

In [ ]:
class CycleVQA:
    def __init__(self, vocab_size, max_question_length=21, image_feature_dim=300, 
                 hidden_dim=512, num_attributes=10):
        self.vocab_size = vocab_size
        self.max_question_length = max_question_length
        self.image_feature_dim = image_feature_dim
        self.hidden_dim = hidden_dim
        self.num_attributes = num_attributes
        
        # Build the model components
        self._build_forward_reasoning()
        self._build_backward_reasoning()
        self._build_cycle_consistent_model()
    
    def _build_forward_reasoning(self):
        """Forward reasoning F: fusion(Q, V) → A"""
        # Question input
        question_input = layers.Input(shape=(self.max_question_length, self.image_feature_dim), 
                                     name='question_input')
        
        # Image input
        image_input = layers.Input(shape=(self.max_question_length, self.image_feature_dim), 
                                  name='image_input')
        
        # Feature fusion
        fused_features = layers.Concatenate(axis=-1)([question_input, image_input])
        fused_features = layers.Dense(self.hidden_dim, activation='relu')(fused_features)
        fused_features = layers.Dropout(0.3)(fused_features)
        
        # Multi-attribute joint analysis
        attribute_features = []
        for i in range(self.num_attributes):
            attr_branch = layers.Dense(self.hidden_dim // 2, activation='relu')(fused_features)
            attr_branch = layers.Dropout(0.3)(attr_branch)
            attr_branch = layers.GlobalAveragePooling1D()(attr_branch)
            attribute_features.append(attr_branch)
        
        # Combine attribute features
        combined_attributes = layers.Concatenate()(attribute_features)
        combined_attributes = layers.Dense(self.hidden_dim, activation='relu')(combined_attributes)
        
        # Answer prediction
        answer_output = layers.Dense(self.vocab_size, activation='softmax', 
                                   name='answer_output')(combined_attributes)
        
        self.forward_model = Model(
            inputs=[question_input, image_input],
            outputs=answer_output,
            name='forward_reasoning'
        )
    
    def _build_backward_reasoning(self):
        """Backward reasoning G: fusion(A, V) → Q"""
        # Answer input (one-hot encoded)
        answer_input = layers.Input(shape=(self.vocab_size,), name='answer_input')
        
        # Image input
        image_input = layers.Input(shape=(self.max_question_length, self.image_feature_dim), 
                                  name='image_input_backward')
        
        # Expand answer to match image dimensions
        expanded_answer = layers.RepeatVector(self.max_question_length)(answer_input)
        
        # Feature fusion
        fused_features = layers.Concatenate(axis=-1)([expanded_answer, image_input])
        fused_features = layers.Dense(self.hidden_dim, activation='relu')(fused_features)
        fused_features = layers.Dropout(0.3)(fused_features)
        
        # Question reconstruction
        question_reconstruction = layers.Dense(self.image_feature_dim, activation='linear',
                                             name='question_reconstruction')(fused_features)
        
        self.backward_model = Model(
            inputs=[answer_input, image_input],
            outputs=question_reconstruction,
            name='backward_reasoning'
        )
    
    def _build_cycle_consistent_model(self):
        """Combine forward and backward reasoning with cycle consistency"""
        # Inputs
        question_input = layers.Input(shape=(self.max_question_length, self.image_feature_dim))
        image_input = layers.Input(shape=(self.max_question_length, self.image_feature_dim))
        
        # Forward reasoning
        answer_pred = self.forward_model([question_input, image_input])
        
        # Backward reasoning (cycle consistency)
        question_recon = self.backward_model([answer_pred, image_input])
        
        # Final model
        self.cycle_model = Model(
            inputs=[question_input, image_input],
            outputs=[answer_pred, question_recon],
            name='cycle_vqa'
        )
    
    def compile(self, optimizer='adam', answer_loss='categorical_crossentropy', 
                cycle_loss='mse', cycle_weight=1.0):
        """Compile the model with appropriate losses"""
        self.cycle_model.compile(
            optimizer=optimizer,
            loss={
                'answer_output': answer_loss,
                'question_reconstruction': cycle_loss
            },
            loss_weights={
                'answer_output': 1.0,
                'question_reconstruction': cycle_weight
            },
            metrics={
                'answer_output': 'accuracy'
            }
        )
    
    def train(self, train_data, val_data=None, epochs=50, batch_size=32, callbacks=None):
        """Train the Cycle-VQA model"""
        if callbacks is None:
            callbacks = [
                tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
                tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
            ]
        
        history = self.cycle_model.fit(
            train_data,
            validation_data=val_data,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )
        
        return history

In [ ]:
# Data preprocessing and loading functions
class VQADataLoader:
    def __init__(self, data_dir, image_dir, json_path, max_length=21, feat_dim=300):
        self.data_dir = data_dir
        self.image_dir = image_dir
        self.max_length = max_length
        self.feat_dim = feat_dim
        
        # Load dataset info
        with open(json_path, 'r') as f:
            self.dataset = json.load(f)
        
        # Create answer vocabulary
        self._create_vocabulary()
        
        # Load pre-trained VGG16 for image feature extraction
        self.vgg_model = tf.keras.applications.VGG16(
            include_top=False, 
            weights='imagenet',
            input_shape=(224, 224, 3)
        )
        self.vgg_model.trainable = False
    
    def _create_vocabulary(self):
        """Create answer vocabulary from the dataset"""
        answer_counts = {}
        for item in self.dataset:
            answer = str(item['answer']).lower().strip()
            answer_counts[answer] = answer_counts.get(answer, 0) + 1
        
        # Get top answers
        sorted_answers = sorted(answer_counts.items(), key=lambda x: x[1], reverse=True)
        self.top_answers = [ans for ans, count in sorted_answers]
        self.answer_to_idx = {ans: idx for idx, ans in enumerate(self.top_answers)}
        self.idx_to_answer = {idx: ans for idx, ans in enumerate(self.top_answers)}
        self.vocab_size = len(self.top_answers)
    
    def _extract_image_features(self, image_path):
        """Extract features from image using VGG16"""
        img = cv2.imread(image_path)
        if img is None:
            return np.zeros((self.max_length, self.feat_dim))
        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        img = tf.keras.applications.vgg16.preprocess_input(img)
        
        # Extract features
        features = self.vgg_model.predict(np.expand_dims(img, axis=0), verbose=0)
        
        # Reshape and project to match question dimensions
        features = features.reshape(-1, features.shape[-1])
        if features.shape[0] > self.max_length:
            features = features[:self.max_length]
        else:
            padding = np.zeros((self.max_length - features.shape[0], features.shape[1]))
            features = np.vstack([features, padding])
        
        # Project to same dimension as word embeddings
        features = self._project_features(features)
        return features
    
    def _project_features(self, features):
        """Project image features to match word embedding dimension"""
        if features.shape[1] != self.feat_dim:
            # Use a simple dense layer for projection (would be trained end-to-end)
            projected = np.dot(features, np.random.randn(features.shape[1], self.feat_dim))
            return projected
        return features
    
    def _process_question(self, question_text, word2vec_model):
        """Process question text into embeddings"""
        tokens = question_text.lower().split()
        embeddings = []
        
        for token in tokens:
            try:
                embeddings.append(word2vec_model[token])
            except:
                # Use zero vector for unknown words
                embeddings.append(np.zeros(self.feat_dim))
        
        # Pad or truncate to max_length
        if len(embeddings) > self.max_length:
            embeddings = embeddings[:self.max_length]
        else:
            padding = [np.zeros(self.feat_dim)] * (self.max_length - len(embeddings))
            embeddings.extend(padding)
        
        return np.array(embeddings)
    
    def load_data(self, word2vec_model, test_split=0.2):
        """Load and preprocess all data"""
        questions = []
        images = []
        answers = []
        
        for item in tqdm(self.dataset, desc="Processing data"):
            # Process question
            question_emb = self._process_question(item['question'], word2vec_model)
            
            # Process image
            image_path = os.path.join(self.image_dir, item['image_name'])
            image_feat = self._extract_image_features(image_path)
            
            # Process answer
            answer_text = str(item['answer']).lower().strip()
            answer_idx = self.answer_to_idx.get(answer_text, 0)
            answer_onehot = tf.keras.utils.to_categorical(
                answer_idx, num_classes=self.vocab_size
            )
            
            questions.append(question_emb)
            images.append(image_feat)
            answers.append(answer_onehot)
        
        # Convert to numpy arrays
        questions = np.array(questions)
        images = np.array(images)
        answers = np.array(answers)
        
        # Split into train and test
        split_idx = int(len(questions) * (1 - test_split))
        
        train_data = (
            [questions[:split_idx], images[:split_idx]],  # Inputs
            [answers[:split_idx], questions[:split_idx]]  # Outputs (answers + cycle consistency)
        )
        
        test_data = (
            [questions[split_idx:], images[split_idx:]],  # Inputs
            [answers[split_idx:], questions[split_idx:]]  # Outputs
        )
        
        return train_data, test_data

In [ ]:
data_loader = VQADataLoader(
        data_dir='/kaggle/working/QA',
        image_dir='/kaggle/input/vqa-rad-visual-question-answering-radiology/VQA_RAD Image Folder',
        json_path='/kaggle/input/vqa-rad-visual-question-answering-radiology/VQA_RAD Dataset Public.json'
    )
    
    # Load word2vec model (assuming it's already downloaded)
import gensim
word2vec_model = gensim.models.KeyedVectors.load_word2vec_format(
    '/kaggle/working/GoogleNews-vectors-negative300.bin', 
    binary=True
    )
    
    # Load data
train_data, test_data = data_loader.load_data(word2vec_model)
    
    # Initialize Cycle-VQA model
cycle_vqa = CycleVQA(
    vocab_size=data_loader.vocab_size,
    max_question_length=21,
    image_feature_dim=300,
    hidden_dim=512,
    num_attributes=10
    )
    
    # Compile model
cycle_vqa.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    answer_loss='categorical_crossentropy',
    cycle_loss='mse',
    cycle_weight=1.0  # Balance between answer accuracy and cycle consistency
    )
    
    # Train model
history = cycle_vqa.train(
    train_data=train_data,
    val_data=test_data,
    epochs=100,
    batch_size=32
    )
    
    # Save model
cycle_vqa.cycle_model.save('/kaggle/working/VQA_Model/cycle_vqa_model.h5')
    
    # Evaluate model
test_loss, test_answer_loss, test_cycle_loss, test_accuracy = cycle_vqa.cycle_model.evaluate(
        test_data[0], test_data[1], verbose=0
    )
    
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Answer Loss: {test_answer_loss:.4f}")
print(f"Cycle Consistency Loss: {test_cycle_loss:.4f}")
    
    return cycle_vqa, history

if __name__ == "__main__":
    model, history = main()